In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1️⃣ Leer los 3 archivos
file_paths = {
    "EICH101": r"C:/Users/diana/OneDrive - Universidad de los Andes/Documentos/9no Semestre/TesisGEB_AlejaYCami/Repositorio/ProyectoGradoGEB/datos/EICH101.xlsx",
    "EICH102": r"C:/Users/diana/OneDrive - Universidad de los Andes/Documentos/9no Semestre/TesisGEB_AlejaYCami/Repositorio/ProyectoGradoGEB/datos/EICH102.xlsx",
    "EICH104": r"C:/Users/diana/OneDrive - Universidad de los Andes/Documentos/9no Semestre/TesisGEB_AlejaYCami/Repositorio/ProyectoGradoGEB/datos/EICH104.xlsx"
}

# Cargar los archivos en dataframes
e1 = pd.read_excel(file_paths["EICH101"])
e2 = pd.read_excel(file_paths["EICH102"])
e3 = pd.read_excel(file_paths["EICH104"])

# 2️⃣ Filtrar registros según ORIG_RAW_VOLUME
e1 = e1[e1["ORIG_RAW_VOLUME"] <= 60]
e2 = e2[e2["ORIG_RAW_VOLUME"] <= 120]
e3 = e3[e3["ORIG_RAW_VOLUME"] <= 350]

# 3️⃣ Convertir EFFECTIVE_DATE a formato de fecha
e1["EFFECTIVE_DATE"] = pd.to_datetime(e1["EFFECTIVE_DATE"], errors='coerce')
e2["EFFECTIVE_DATE"] = pd.to_datetime(e2["EFFECTIVE_DATE"], errors='coerce')
e3["EFFECTIVE_DATE"] = pd.to_datetime(e3["EFFECTIVE_DATE"], errors='coerce')

# Agregar columna de empresa e identificarla con One-Hot Encoding
e1["empresa"] = "EICH101"
e2["empresa"] = "EICH102"
e3["empresa"] = "EICH104"

# Unificar los tres DataFrames en uno solo
df = pd.concat([e1, e2, e3], ignore_index=True)

# Aplicar One-Hot Encoding a la columna "empresa"
df = pd.get_dummies(df, columns=["empresa"])

# 4️⃣ Eliminar registros con valores nulos, duplicados y negativos
df = df.dropna()  # Eliminar registros con valores nulos
df = df.drop_duplicates()  # Eliminar duplicados
# Seleccionar solo columnas numéricas antes de eliminar valores negativos
numeric_columns = df.select_dtypes(include=["number"]).columns
df = df[df[numeric_columns].ge(0).all(axis=1)]


# 5️⃣ Aplicar normalización a ORIG_RAW_VOLUME usando percentil 99
global_max = np.percentile(df["ORIG_RAW_VOLUME"], 99)
df["ORIG_RAW_VOLUME_NORM"] = df["ORIG_RAW_VOLUME"] / global_max
df["ORIG_RAW_VOLUME_NORM"] = np.clip(df["ORIG_RAW_VOLUME_NORM"], 0, 1)  # Asegurar que esté entre 0 y 1


In [23]:
print(df.head())

   ORIG_STD_VOLUME  STD_VOLUME  ORIG_TEMPERATURE  TEMPERATURE   PRESSURE  \
0         0.000000    0.000000         23.304819    23.304819  17.686537   
1        17.516075   17.516081         23.449533    23.449533  17.705580   
2        19.025777   19.025784         23.625006    23.625006  17.690292   
3        14.820577   14.820582         23.636658    23.636658  17.708681   
4        14.184913   14.184918         23.636869    23.636869  17.692707   

   ORIG_PRESSURE  ORIG_RAW_VOLUME  RAW_VOLUME      EFFECTIVE_DATE  \
0      17.686537            0.000       0.000 2018-09-03 09:00:00   
1      17.705580           25.875      25.875 2018-09-03 10:00:00   
2      17.690292           28.125      28.125 2018-09-03 11:00:00   
3      17.708681           21.875      21.875 2018-09-03 12:00:00   
4      17.692707           21.000      21.000 2018-09-03 13:00:00   

   empresa_EICH101  empresa_EICH102  empresa_EICH104  ORIG_RAW_VOLUME_NORM  
0             True            False            Fals

In [24]:
import os
import pandas as pd
import numpy as np
import pywt
import matplotlib.pyplot as plt
import math

# Crear la carpeta donde se guardarán las imágenes
output_folder = "C:/Users/diana/OneDrive - Universidad de los Andes/Documentos/9no Semestre/TesisGEB_AlejaYCami/Repositorio/ProyectoGradoGEB/modelos/escalogramas"
os.makedirs(output_folder, exist_ok=True)

# Escalas para la Transformada Wavelet Continua (CWT)
scales = np.arange(1, 128)

# Lista de empresas (columnas One-Hot Encoding)
empresas = ['empresa_EICH101', 'empresa_EICH102', 'empresa_EICH104']

# Recorrer el archivo secuencialmente
df_sorted = df.sort_values(by="EFFECTIVE_DATE")

for empresa in empresas:
    # Filtrar solo las filas donde la empresa tiene valor 1 en One-Hot Encoding
    df_empresa = df_sorted[df_sorted[empresa] == 1]

    # Obtener los años únicos en el DataFrame
    years = df_empresa['EFFECTIVE_DATE'].dt.year.unique()

    for year in years:
        df_year = df_empresa[df_empresa['EFFECTIVE_DATE'].dt.year == year]

        # Obtener las semanas únicas ordenadas
        weeks = sorted(df_year['EFFECTIVE_DATE'].dt.isocalendar().week.unique())

        for week in weeks:
            df_week = df_year[df_year['EFFECTIVE_DATE'].dt.isocalendar().week == week]

            if not df_week.empty:
                # Extraer la señal de la semana (manteniendo valores originales sin escalar)
                signal_week = df_week['ORIG_RAW_VOLUME_NORM'].values

                # Aplicar la Transformada Wavelet Continua (CWT) con Morlet
                coefficients, frequencies = pywt.cwt(signal_week, scales, 'morl')

                # Crear la figura del escalograma
                plt.figure(figsize=(10, 6))
                plt.imshow(np.abs(coefficients), aspect="auto", cmap="jet",
                           extent=[0, len(signal_week), scales[-1], scales[0]])
                plt.colorbar(label="Magnitud")
                plt.title(f"Escalograma - {empresa} - Año {year} - Semana {week}", fontsize=12, fontweight="bold")
                plt.xlabel("Horas de la Semana")
                plt.ylabel("Escala")

                # Guardar la imagen
                filename = f"{empresa}_Año{year}_Semana{week}.png".replace(" ", "_")
                filepath = os.path.join(output_folder, filename)
                plt.savefig(filepath, dpi=300, bbox_inches="tight")
                plt.close()

# Listar los archivos generados
generated_files = os.listdir(output_folder)
print("Ejemplo de archivos generados:", generated_files[:10])  # Mostrar solo los primeros 10 archivos como muestra

Ejemplo de archivos generados: ['empresa_EICH101_Año2018_Semana1.png', 'empresa_EICH101_Año2018_Semana36.png', 'empresa_EICH101_Año2018_Semana37.png', 'empresa_EICH101_Año2018_Semana38.png', 'empresa_EICH101_Año2018_Semana39.png', 'empresa_EICH101_Año2018_Semana40.png', 'empresa_EICH101_Año2018_Semana41.png', 'empresa_EICH101_Año2018_Semana42.png', 'empresa_EICH101_Año2018_Semana43.png', 'empresa_EICH101_Año2018_Semana44.png']
